## 2. Reactor data and model assumptions

The reactor considered in this assignment is a tubular reactor in which component A is converted into component B. For the first implementation, the reaction is treated as irreversible and first order.

$$
\mathrm{A \rightarrow B}
$$

The reaction rate used in the code is:

$$
r = k c_A
$$

The reactor length is 10 m and the reactor diameter is 32 mm. The upstream pipe diameter is 150 mm. The volumetric flow rate is 1 L/s, which is converted to 0.001 m3/s in the code. The inlet concentration of component A is 100 mol/m3, while the inlet concentration of component B is set to 0 mol/m3.

The reactor cross-sectional area is calculated from the reactor diameter.

$$
A_R = \frac{\pi d_R^2}{4}
$$

The upstream pipe cross-sectional area is calculated in the same way.

$$
A_{up} = \frac{\pi d_{up}^2}{4}
$$

The superficial velocity in the reactor is calculated using the volumetric flow rate and the reactor cross-sectional area.

$$
u = \frac{\dot{V}}{A_R}
$$

The superficial velocity in the upstream pipe is calculated as:

$$
u_{up} = \frac{\dot{V}}{A_{up}}
$$

The mean residence time in the reactor is:

$$
\tau = \frac{L}{u}
$$

Using the values implemented in the code, the calculated reactor quantities are:

| Quantity | Symbol | Value | Unit |
|---|---:|---:|---|
| Reactor length | L | 10.0000 | m |
| Reactor diameter | d_R | 0.0320 | m |
| Upstream pipe diameter | d_up | 0.1500 | m |
| Volumetric flow rate | V_dot | 0.0010 | m3/s |
| Reactor cross-sectional area | A_R | 8.0425e-4 | m2 |
| Upstream cross-sectional area | A_up | 1.7671e-2 | m2 |
| Reactor superficial velocity | u | 1.2434 | m/s |
| Upstream superficial velocity | u_up | 0.0566 | m/s |
| Mean residence time | tau | 8.0425 | s |
| Inlet concentration of A | c_A,in | 100.0000 | mol/m3 |
| Inlet concentration of B | c_B,in | 0.0000 | mol/m3 |
| First-order rate constant | k | 0.2000 | 1/s |

The superficial velocity inside the reactor is much higher than the upstream velocity because the reactor diameter is significantly smaller than the upstream pipe diameter. Therefore, the same volumetric flow rate passes through a smaller cross-sectional area in the reactor.

The following assumptions are used in this model:

| Assumption | Meaning |
|---|---|
| Steady-state operation | Concentrations are independent of time. |
| Isothermal system | Temperature effects are not included in this first model. |
| Irreversible first-order reaction | The reaction rate is proportional to the concentration of A. |
| Constant volumetric flow rate | Density and volume changes are neglected. |
| One-dimensional axial model | Concentration varies only along the reactor length z. |
| No radial concentration gradients | Radial mixing is assumed to be sufficient. |
| No side reactions | Only the conversion of A to B is considered. |
| Constant axial dispersion coefficient for each Bo | For each selected Bodenstein number, D_ax is constant along the reactor. |

The value k = 0.2 1/s is used as a reference value because no numerical value of the rate constant is specified in the task statement. This value gives an intermediate outlet conversion of about 0.80 in the ideal PFR. This makes it easier to compare ideal plug-flow behaviour, axial-dispersion behaviour and CSTR-cascade behaviour.

## 3. Ideal PFR model and solver validation

The ideal plug-flow reactor is first implemented as a reference model. In this model, axial dispersion and backmixing are neglected. The only transport mechanism along the reactor is convection, and the reaction takes place as the fluid moves through the reactor.

For the reaction:

$$
\mathrm{A \rightarrow B}
$$

and the first-order rate law:

$$
r = k c_A
$$

the material balance for component A is:

$$
u \frac{dc_A}{dz} = -k c_A
$$

The material balance for component B is:

$$
u \frac{dc_B}{dz} = k c_A
$$

These equations are solved as an initial value problem because the inlet concentrations are known at the beginning of the reactor.

$$
c_A(0) = c_{A,in}
$$

$$
c_B(0) = c_{B,in}
$$

In the code, this ideal PFR model is implemented in the function `pfr_ivp_rhs`. The function calculates the reaction rate and returns the derivatives of c_A and c_B with respect to the axial coordinate z. The system is solved numerically using `solve_ivp`.

For a first-order irreversible reaction in an ideal PFR, the concentration of A also has an analytical solution.

$$
c_A(z) = c_{A,in}\exp\left(-\frac{kz}{u}\right)
$$

The conversion of A is calculated as:

$$
X_A = \frac{c_{A,in} - c_A}{c_{A,in}}
$$

At the reactor outlet, the ideal PFR conversion can be written using the mean residence time.

$$
X_{A,out} = 1 - \exp(-k\tau)
$$

With the values used in the code:

$$
k = 0.2 \ \mathrm{s^{-1}}
$$

$$
\tau = 8.0425 \ \mathrm{s}
$$

The first-order Damköhler number is:

$$
Da_I = k \tau
$$

$$
Da_I = 0.2 \cdot 8.0425 = 1.6085
$$

Therefore, the ideal PFR outlet conversion is:

$$
X_{A,out} = 1 - \exp(-1.6085)
$$

$$
X_{A,out} \approx 0.7998
$$

The numerical result from the final summary gives:

| Quantity | Value |
|---|---:|
| c_A,out | 20.0189 mol/m3 |
| c_B,out | 79.9811 mol/m3 |
| X_A,out | 0.7998 |

The ideal PFR concentration profiles show that c_A decreases continuously along the reactor length, while c_B increases correspondingly. This is expected for the reaction A to B because the consumption of A directly forms B.

The numerical solution is then compared with the analytical solution for c_A. The plotted numerical and analytical curves overlap almost exactly. This confirms that the `solve_ivp` implementation, the first-order rate law and the material balances are consistent.

The conversion profile increases smoothly from zero at the inlet to approximately 0.80 at the outlet. This ideal PFR solution is used as the reference case for the later axial-dispersion model. In the limit of very high Bodenstein number, axial dispersion becomes very small, so the BVP solution should approach this ideal PFR result.

## 4. Axial-dispersion BVP model

After validating the ideal PFR model, axial dispersion is included in the reactor model. In contrast to the ideal PFR, the axial-dispersion model accounts for backmixing in the axial direction. Therefore, the concentration profile is influenced by convection, reaction and axial dispersion.

For a general component i, the axial-dispersion balance can be written as:

$$
u \frac{dc_i}{dz}
=
D_{ax}\frac{d^2c_i}{dz^2}
+
\sum_j \nu_{i,j} r_j
$$

For the reaction:

$$
\mathrm{A \rightarrow B}
$$

with first-order kinetics:

$$
r = k c_A
$$

the balance for component A becomes:

$$
u \frac{dc_A}{dz}
=
D_{ax}\frac{d^2c_A}{dz^2}
-
k c_A
$$

The balance for component B becomes:

$$
u \frac{dc_B}{dz}
=
D_{ax}\frac{d^2c_B}{dz^2}
+
k c_A
$$

The axial-dispersion model contains second derivatives with respect to the reactor coordinate z. Therefore, it cannot be solved only from inlet initial values. Instead, boundary conditions are required at both the inlet and the outlet. For this reason, the axial-dispersion model is implemented as a boundary value problem using `solve_bvp`.

To solve the second-order differential equations with `solve_bvp`, the model is rewritten as a first-order system. The following variables are introduced:

$$
y_0 = c_A
$$

$$
y_1 = \frac{dc_A}{dz}
$$

$$
y_2 = c_B
$$

$$
y_3 = \frac{dc_B}{dz}
$$

The corresponding first-order system is:

$$
\frac{dy_0}{dz} = y_1
$$

$$
\frac{dy_1}{dz}
=
\frac{u y_1 + k y_0}{D_{ax}}
$$

$$
\frac{dy_2}{dz} = y_3
$$

$$
\frac{dy_3}{dz}
=
\frac{u y_3 - k y_0}{D_{ax}}
$$

This is exactly the structure implemented in the function `dispersion_bvp_rhs`. The function returns the first derivatives of the four state variables with respect to the axial coordinate.

The axial dispersion coefficient is related to the Bodenstein number by:

$$
Bo = \frac{uL}{D_{ax}}
$$

Therefore:

$$
D_{ax} = \frac{uL}{Bo}
$$

A high Bodenstein number corresponds to a small axial dispersion coefficient. This means that convective transport dominates and the reactor approaches ideal plug-flow behaviour. A low Bodenstein number corresponds to a larger axial dispersion coefficient, meaning stronger axial backmixing and stronger deviation from the ideal PFR.

The inlet boundary condition is written as a Danckwerts boundary condition. For component A:

$$
c_A(0) - \frac{D_{ax}}{u}\frac{dc_A}{dz}\bigg|_{z=0}
=
c_{A,in}
$$

For component B:

$$
c_B(0) - \frac{D_{ax}}{u}\frac{dc_B}{dz}\bigg|_{z=0}
=
c_{B,in}
$$

At the reactor outlet, a zero-gradient boundary condition is applied:

$$
\frac{dc_A}{dz}\bigg|_{z=L} = 0
$$

$$
\frac{dc_B}{dz}\bigg|_{z=L} = 0
$$

The Danckwerts inlet condition allows the concentration directly inside the reactor at z = 0 to differ from the feed concentration. This difference is caused by axial dispersion and is later evaluated as the inlet concentration jump. The outlet zero-gradient condition assumes that there is no further axial concentration gradient at the reactor exit.

In the code, these four boundary conditions are implemented in the function `dispersion_bc`. Together, `dispersion_bvp_rhs` and `dispersion_bc` define the complete axial-dispersion BVP.

## 5. High-Bodenstein-number validation of the BVP

To validate the axial-dispersion BVP implementation, the model is first solved at a very high Bodenstein number.

$$
Bo = 10000
$$

The axial dispersion coefficient is calculated from:

$$
D_{ax} = \frac{uL}{Bo}
$$

Since the Bodenstein number is very large, the axial dispersion coefficient becomes very small. Physically, this means that convective transport dominates over axial backmixing. Therefore, the axial-dispersion model should approach the ideal PFR model.

In the code, the BVP solution at \(Bo = 10000\) is compared with the ideal PFR IVP solution for \(c_A\), \(c_B\), and \(X_A\). The concentration profile of component A from the dispersion model overlaps almost exactly with the ideal PFR result. The same behaviour is observed for component B. This confirms that the axial-dispersion model correctly approaches the plug-flow limit when axial dispersion becomes negligible.

The conversion profiles also overlap almost completely. The relative difference in outlet conversion between the high-Bodenstein-number BVP and the ideal PFR IVP is:

$$
\mathrm{relative \ difference} = 0.01\%
$$

This very small difference shows that the BVP implementation is numerically consistent with the ideal PFR model in the high-\(Bo\) limit.

The Danckwerts inlet boundary condition also causes a small difference between the feed concentration and the concentration directly inside the reactor at \(z = 0\). From the code output:

| Quantity | Value |
|---|---:|
| Feed concentration \(c_{A,in}\) | 100.0000 mol/m3 |
| In-reactor inlet concentration \(c_A(z=0)\) | 99.9839 mol/m3 |
| Inlet concentration jump | 0.0161 mol/m3 |
| Relative inlet concentration jump | 0.02% |

The inlet concentration jump is very small because \(Bo = 10000\) corresponds to weak axial dispersion. This is expected, because in the ideal plug-flow limit there should be no significant backmixing at the inlet.

Overall, the high-\(Bo\) validation confirms that the axial-dispersion BVP is implemented correctly. When axial dispersion is made very small, the BVP result approaches the ideal PFR result for both concentration and conversion profiles.

## 6. Inlet concentration jump for different Bodenstein numbers

The Danckwerts inlet boundary condition allows the concentration directly inside the reactor at \(z = 0\) to be different from the feed concentration. This difference is called the inlet concentration jump.

For component A, the Danckwerts inlet condition is:

$$
c_A(0) - \frac{D_{ax}}{u}\frac{dc_A}{dz}\bigg|_{z=0}
=
c_{A,in}
$$

Therefore, if axial dispersion is significant, the concentration inside the reactor at the inlet does not have to be exactly equal to the feed concentration. This is a characteristic feature of the axial-dispersion model and does not occur in the ideal PFR model.

In the code, the inlet concentration jump is calculated for three different Bodenstein numbers:

$$
Bo = 100,\ 1000,\ 10000
$$

The results are:

| Bo | c_A(z=0) / mol/m3 | Inlet jump / % |
|---:|---:|---:|
| 100 | 98.4413 | 1.5587 |
| 1000 | 99.8397 | 0.1603 |
| 10000 | 99.9839 | 0.0161 |

The results show that the inlet concentration jump decreases strongly as the Bodenstein number increases. At \(Bo = 100\), the concentration directly inside the reactor is 98.4413 mol/m3, giving an inlet jump of 1.5587%. At \(Bo = 10000\), the concentration inside the reactor is almost equal to the feed concentration, and the inlet jump is only 0.0161%.

This behaviour is physically reasonable because a larger Bodenstein number corresponds to weaker axial dispersion. When axial dispersion is weak, the reactor behaves more like an ideal PFR and the inlet concentration inside the reactor approaches the feed concentration. When the Bodenstein number is lower, axial backmixing becomes stronger, and the difference between the feed concentration and the in-reactor inlet concentration becomes larger.

Thus, the inlet concentration jump provides a useful numerical indicator of the strength of axial backmixing in the Danckwerts boundary condition.

## 7. Full Bodenstein-number study

After the inlet-jump check, the influence of axial dispersion is investigated over a wider range of Bodenstein numbers. The values used in the code are:

$$
Bo = 5,\ 10,\ 20,\ 50,\ 100,\ 1000,\ 10000
$$

For each Bodenstein number, the axial dispersion coefficient is calculated using:

$$
D_{ax} = \frac{uL}{Bo}
$$

Therefore, a low Bodenstein number corresponds to a high axial dispersion coefficient, while a high Bodenstein number corresponds to a low axial dispersion coefficient.

The calculated results are:

| Bo | D_ax / m2/s | c_A(z=0) / mol/m3 | Inlet jump / % | c_A,out / mol/m3 | c_B,out / mol/m3 | X_A,out |
|---:|---:|---:|---:|---:|---:|---:|
| 5 | 2.4868 | 79.6212 | 20.3788 | 26.6341 | 73.3659 | 0.7337 |
| 10 | 1.2434 | 87.6443 | 12.3557 | 24.0475 | 75.9525 | 0.7595 |
| 20 | 0.6217 | 93.0383 | 6.9617 | 22.2823 | 77.7177 | 0.7772 |
| 50 | 0.2487 | 96.9747 | 3.0253 | 20.9979 | 79.0021 | 0.7900 |
| 100 | 0.1243 | 98.4413 | 1.5587 | 20.5221 | 79.4779 | 0.7948 |
| 1000 | 0.0124 | 99.8397 | 0.1603 | 20.0705 | 79.9295 | 0.7993 |
| 10000 | 0.0012 | 99.9839 | 0.0161 | 20.0240 | 79.9760 | 0.7998 |

The results show that the outlet conversion increases as the Bodenstein number increases. At \(Bo = 5\), the outlet conversion is 0.7337. At \(Bo = 10000\), the outlet conversion is 0.7998, which is almost identical to the ideal PFR conversion. This confirms that the axial-dispersion model approaches ideal plug-flow behaviour when the Bodenstein number becomes very large.

The concentration profiles also show the effect of axial dispersion. For low Bodenstein numbers, the concentration of A at the reactor inlet is already significantly lower than the feed concentration. For example, at \(Bo = 5\), the in-reactor inlet concentration is only 79.6212 mol/m3. This is caused by strong axial backmixing and the Danckwerts inlet boundary condition.

As the Bodenstein number increases, the \(c_A\) profile becomes closer to the ideal PFR profile. At \(Bo = 10000\), the axial-dispersion profile almost overlaps with the ideal PFR profile. The same trend is observed for component B and for the conversion profile.

The outlet-conversion plot shows that the strongest change occurs at low Bodenstein numbers. Increasing \(Bo\) from 5 to 100 causes a clear increase in outlet conversion. However, increasing \(Bo\) further from 1000 to 10000 causes only a very small change. This means that once axial dispersion becomes sufficiently small, the reactor behaviour is already close to the ideal PFR limit.

The inlet concentration jump also decreases strongly with increasing \(Bo\). At \(Bo = 5\), the inlet jump is 20.3788%, while at \(Bo = 10000\), it is only 0.0161%. This confirms that lower \(Bo\) values represent stronger backmixing at the reactor inlet.

The dispersive flux of A is calculated in the code as:

$$
J_{disp,A} = -D_{ax}\frac{dc_A}{dz}
$$

The maximum absolute dispersive flux is highest at low Bodenstein number. The output shows that the maximum value decreases from 25.3390 mol/m2/s at \(Bo = 5\) to only 0.0200 mol/m2/s at \(Bo = 10000\). In all cases, the maximum dispersive flux occurs at \(z = 0\). This is reasonable because the concentration gradient is strongest near the reactor inlet.

Overall, the Bodenstein-number study shows that axial dispersion reduces the plug-flow character of the reactor. Stronger axial dispersion, represented by lower \(Bo\), causes a larger inlet concentration jump, stronger dispersive flux and lower outlet conversion. Higher \(Bo\) values reduce these effects and move the reactor behaviour closer to the ideal PFR.

## 8. Theoretical and practical boundary cases

The axial-dispersion model can represent different reactor behaviours depending on the value of the Bodenstein number. Since the Bodenstein number is defined as:

$$
Bo = \frac{uL}{D_{ax}}
$$

the magnitude of \(Bo\) directly describes the relative importance of convective transport compared with axial dispersion.

A very high Bodenstein number means that \(D_{ax}\) is very small. In this case, convection dominates and axial backmixing becomes negligible. The axial-dispersion model then approaches the ideal plug-flow reactor.

$$
Bo \rightarrow \infty
$$

$$
D_{ax} \rightarrow 0
$$

This is the theoretical plug-flow limit. In this limit, the second derivative term in the axial-dispersion model disappears and the ideal PFR balance is recovered.

For high but finite Bodenstein numbers, the reactor still behaves close to an ideal PFR, but a small amount of axial dispersion remains. This is seen in the high-\(Bo\) validation, where \(Bo = 10000\) gives almost the same outlet conversion as the ideal PFR.

For low Bodenstein numbers, the axial dispersion coefficient becomes larger.

$$
Bo \downarrow
$$

$$
D_{ax} \uparrow
$$

This means that axial backmixing becomes stronger. Stronger backmixing smooths the concentration gradients and causes a larger deviation from ideal plug-flow behaviour. In the simulation results, low \(Bo\) values also give a larger inlet concentration jump and lower outlet conversion.

For very large \(D_{ax}\), the concentration profile becomes flatter and the reactor tends toward mixed-reactor-like behaviour. This does not mean that the reactor is exactly a CSTR, but the strong axial mixing reduces the plug-flow character of the system.

The boundary cases used in the code are summarized as follows:

| Case | Mathematical meaning | Expected reactor behaviour | Practical interpretation |
|---|---|---|---|
| \(Bo \rightarrow \infty\) | \(D_{ax} \rightarrow 0\) | Axial dispersion disappears and the model approaches ideal plug flow. | Very weak backmixing; convection dominates over axial mixing. |
| High \(Bo\) | Small \(D_{ax}\) | The axial-dispersion model remains close to the ideal PFR. | Typical of strongly convective tubular reactors with limited axial mixing. |
| Low \(Bo\) | Large \(D_{ax}\) | Stronger backmixing smooths concentration gradients and causes larger deviation from PFR behaviour. | Axial mixing becomes important and the reactor becomes less plug-flow-like. |
| Very large \(D_{ax}\) | Very strong axial dispersion | The reactor tends toward mixed-reactor-like behaviour. | The axial concentration profile becomes flatter and the reactor behaves more like a mixed system. |
| \(D_{ax} \rightarrow 0\) | No dispersion term | The second derivative term disappears and the ideal PFR balance is recovered. | This is the theoretical plug-flow limit. |

These boundary cases help interpret the numerical results. The ideal PFR represents the no-dispersion limit, while low-\(Bo\) axial-dispersion cases represent stronger backmixing. The CSTR cascade is then used as a second non-ideal reactor model to compare with the dispersion model.

## 9. CSTR cascade model

After the axial-dispersion model, a CSTR cascade is implemented as another way to describe non-ideal reactor behaviour. A CSTR cascade consists of several ideally mixed tanks connected in series. Each tank is treated as a steady-state CSTR.

In the code, the function `cstr_cascade` is used to calculate the concentration of A and B at the outlet of each tank. The total residence time is divided equally between the tanks.

$$
\tau_i = \frac{\tau}{N}
$$

For a first-order reaction in one CSTR, the steady-state balance for component A is:

$$
c_{A,i-1} - c_{A,i} = k \tau_i c_{A,i}
$$

Rearranging gives:

$$
c_{A,i} = \frac{c_{A,i-1}}{1 + k\tau_i}
$$

This is the equation used in the loop of the CSTR-cascade code.

For component B, the code uses the stoichiometric mass balance that the amount of A consumed becomes B:

$$
c_{B,i} = c_{B,i-1} + \left(c_{A,i-1} - c_{A,i}\right)
$$

The CSTR cascade is first evaluated for:

$$
N = 10
$$

The residence time in each tank is therefore:

$$
\tau_i = \frac{8.0425}{10}
$$

$$
\tau_i = 0.80425 \ \mathrm{s}
$$

For \(N = 10\), the final outlet values are:

| Quantity | Value |
|---|---:|
| \(c_{A,out}\) | 22.5030 mol/m3 |
| \(c_{B,out}\) | 77.4970 mol/m3 |
| \(X_{A,out}\) | 0.7750 |

The CSTR cascade concentration profile is plotted as a step profile because each tank is ideally mixed. Therefore, the concentration is uniform inside each tank and changes only from one tank outlet to the next. This is different from the ideal PFR and axial-dispersion models, which give continuous concentration profiles along the reactor length.

The conversion profile for the CSTR cascade increases from zero at the inlet to approximately 0.775 at the outlet for \(N = 10\). This conversion is lower than the ideal PFR conversion of approximately 0.7998. This is expected because mixing reduces the plug-flow character of the reactor and generally lowers conversion for a positive-order reaction at the same total residence time.

## 10. Effect of number of CSTRs

The number of CSTRs in series controls how close the cascade behaves to an ideal plug-flow reactor. A single CSTR is completely mixed and gives the strongest deviation from plug flow. As the number of tanks increases, each tank becomes smaller and the concentration changes more gradually along the reactor length.

In the code, the effect of the number of CSTRs is investigated for:

$$
N = 1,\ 2,\ 5,\ 10,\ 50
$$

The same total residence time is used in all cases:

$$
\tau = 8.0425 \ \mathrm{s}
$$

For a cascade of \(N\) identical CSTRs with first-order kinetics, the outlet concentration of A can be written as:

$$
c_{A,out}
=
\frac{c_{A,in}}{\left(1 + \frac{k\tau}{N}\right)^N}
$$

The corresponding outlet conversion is:

$$
X_{A,out}
=
1 -
\frac{1}{\left(1 + \frac{k\tau}{N}\right)^N}
$$

Using the values from the code, the outlet results are:

| Number of CSTRs \(N\) | \(c_{A,out}\) / mol/m3 | \(c_{B,out}\) / mol/m3 | \(X_{A,out}\) |
|---:|---:|---:|---:|
| 1 | 38.3363 | 61.6637 | 0.6166 |
| 2 | 30.7190 | 69.2810 | 0.6928 |
| 5 | 24.7935 | 75.2065 | 0.7521 |
| 10 | 22.5030 | 77.4970 | 0.7750 |
| 50 | 20.5324 | 79.4676 | 0.7947 |

The results show that the outlet conversion increases as the number of CSTRs increases. For \(N = 1\), the reactor behaves as a single mixed tank and the outlet conversion is only 0.6166. For \(N = 10\), the outlet conversion increases to 0.7750. For \(N = 50\), the outlet conversion is 0.7947, which is close to the ideal PFR outlet conversion of 0.7998.

This behaviour is physically expected. A single CSTR has strong internal mixing, so the reactant concentration inside the reactor is close to the outlet concentration. This lowers the effective reaction rate for a first-order reaction. When more CSTRs are placed in series, the concentration decreases stepwise along the reactor instead of being fully mixed over the entire volume. Therefore, the reactor behaves more like a plug-flow reactor.

The plot comparing \(N = 1, 2, 5, 10,\) and \(50\) confirms this trend. The step profiles become smoother and approach the ideal PFR curve as \(N\) increases. Thus, the CSTR cascade provides a simple way to approximate plug-flow behaviour using a series of mixed reactors.

## 11. Comparison of CSTR cascade with dispersion model at three distinct points

The assignment requires a comparison between the CSTR cascade and the axial-dispersion model at three distinct points along the reactor. In the code, this comparison is performed using:

$$
N = 10
$$

for the CSTR cascade and:

$$
Bo = 20
$$

for the axial-dispersion model.

The comparison is carried out at the following axial positions:

$$
z/L = 0.25,\ 0.50,\ 1.00
$$

These correspond to:

$$
z = 2.5\ \mathrm{m},\ 5.0\ \mathrm{m},\ 10.0\ \mathrm{m}
$$

The comparison results are:

| z / m | z/L | c_A dispersion / mol/m3 | c_A CSTR cascade / mol/m3 | Difference c_A / mol/m3 | X_A dispersion | X_A CSTR cascade | Difference X_A |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 2.5 | 0.25 | 64.0000 | 69.0664 | 5.0664 | 0.3600 | 0.3093 | -0.0507 |
| 5.0 | 0.50 | 44.0249 | 47.4373 | 3.4125 | 0.5598 | 0.5256 | -0.0341 |
| 10.0 | 1.00 | 22.2823 | 22.5030 | 0.2207 | 0.7772 | 0.7750 | -0.0022 |

The CSTR cascade gives a higher concentration of A than the dispersion model at all three comparison points. Therefore, the conversion predicted by the CSTR cascade is lower than the conversion predicted by the dispersion model.

The largest difference occurs near the first comparison point at \(z/L = 0.25\). At this position, the CSTR cascade conversion is lower by about 0.0507. This happens because the CSTR cascade is represented by stepwise mixed zones. In each tank, the concentration is uniform, so the concentration profile changes discontinuously from one tank outlet to the next.

At \(z/L = 0.50\), the difference becomes smaller. The conversion difference is about 0.0341. At the reactor outlet, the difference is very small. The outlet conversion is 0.7772 for the dispersion model and 0.7750 for the CSTR cascade.

This shows that the two non-ideal reactor models are not identical along the full reactor length, but they can give similar outlet predictions when a suitable number of CSTRs and a suitable Bodenstein number are chosen. The axial-dispersion model gives a continuous concentration profile, while the CSTR cascade gives a stepwise concentration profile. Both models describe deviations from ideal plug flow, but they represent mixing in different mathematical ways.